# 02 - Relational Modelling & SQL Analysis

**Input**: `clean_transactions.parquet` (notebook 01), 820,947 rows, 5,880 customers

**Output**: `retail.db` (SQLite) and aggregated CSV extracts for BI reporting

---

## Business Context & Objectives

Notebook 01 produced a clean but flat table, in which every row repeats its customer's country and its product's description. This notebook normalises that table into a relational schema, enforces integrity at the database level, and answers the reporting questions not covered by segmentation (03) or retention (04): revenue over time, customer ranking, and product performance.

### Objectives

1. Decompose the flat file into four normalised tables with declared primary and foreign keys.
2. Resolve, and document, the conflicts that prevent normalisation.
3. Verify that the declared constraints are enforced by the database.
4. Produce the aggregates consumed by the BI dashboard.

# 0. Setup & Loading

In [1]:
import sqlite3
import pandas as pd
from pathlib import Path
from IPython.display import display, Markdown

PROC = Path("../data/processed")
DB_PATH = PROC / "retail.db"

pd.set_option("display.max_columns", None)

In [2]:
df = pd.read_parquet(PROC / "clean_transactions.parquet")

display(Markdown(f"""
### Analytical base loaded
* **Rows:** `{len(df):,}`
* **Customers:** `{df['Customer ID'].nunique():,}`
* **Products:** `{df['StockCode'].nunique():,}`
* **Invoices:** `{df['Invoice'].nunique():,}`
"""))
df.head()


### Analytical base loaded
* **Rows:** `820,947`
* **Customers:** `5,880`
* **Products:** `4,638`
* **Invoices:** `43,929`


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,SourceSheet,StockCodeUpper,IsNonProduct,IsCancellation,LineRevenue
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,Year 2009-2010,85048,False,False,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,Year 2009-2010,79323P,False,False,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,Year 2009-2010,79323W,False,False,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,Year 2009-2010,22041,False,False,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,Year 2009-2010,21232,False,False,30.0


**Results.** The analytical base loads as expected: 820,947 rows covering 5,880 customers, 4,638 distinct products and 43,929 invoices. These figures match the export from notebook 01, so no data was lost in transit.

# 1. Relational Schema Design

| Table | Primary key | Foreign keys | Content |
|---|---|---|---|
| `customers` | `customer_id` | (none) | Country of record |
| `products` | `stock_code` | (none) | Product label |
| `invoices` | `invoice_no` | `customer_id` | Date, cancellation flag |
| `invoice_lines` | `line_id` | `invoice_no`, `stock_code` | Quantity, price, line revenue |

Normalisation assumes one country per customer and one description per stock code. The flat file guarantees neither, so both assumptions are checked before the tables are built.

## 1.1 Normalisation Conflicts

In [3]:
countries_per_customer = df.groupby("Customer ID")["Country"].nunique()
labels_per_product = df.groupby("StockCode")["Description"].nunique()

display(Markdown(f"""
### Obstacles to normalisation
* Customers linked to **multiple countries**: `{(countries_per_customer > 1).sum():,}` out of {len(countries_per_customer):,}
* Stock codes with **multiple descriptions**: `{(labels_per_product > 1).sum():,}` out of {len(labels_per_product):,}
"""))

display(Markdown("**Example of a stock code with multiple labels:**"))
example = labels_per_product[labels_per_product > 1].index[0]
display(df.loc[df["StockCode"] == example, ["StockCode", "Description"]]
          .value_counts().head(5).to_frame("Occurrences"))


### Obstacles to normalisation
* Customers linked to **multiple countries**: `12` out of 5,880
* Stock codes with **multiple descriptions**: `598` out of 4,638


**Example of a stock code with multiple labels:**

Occurrences
StockCode Description                                 
15058A    BLUE WHITE SPOTS GARDEN PARASOL           58
          BLUE POLKADOT GARDEN PARASOL              55

**Results.** 12 customers out of 5,880 (0.2%) appear under more than one country. 598 stock codes out of 4,638 (12.9%) carry more than one description.

These are label variants rather than mis-assignments. `15058A` appears 58 times as *BLUE WHITE SPOTS GARDEN PARASOL* and 55 times as *BLUE POLKADOT GARDEN PARASOL*, two ways of describing the same polka-dot parasol.

The near-tie (58 against 55) matters. A "most frequent" rule alone would be close to arbitrary here, and could return a different winner depending on row order. The rule therefore needs a deterministic tie-breaker.

### Remediation rules

| Conflict | Scale | Rule | Rationale |
|---|---|---|---|
| Customer with multiple countries | 12 / 5,880 | Most frequent country, ties broken by most recent order | Negligible volume; the modal country reflects the usual place of business |
| Stock code with multiple descriptions | 598 / 4,638 | Most frequent description, ties broken alphabetically | The description is a display label, not an identifier. `StockCode` carries product identity, and alphabetical tie-breaking makes the result reproducible |

In [4]:
# Country: most frequent, ties broken by most recent order
country_res = (
    df.groupby(["Customer ID", "Country"])
      .agg(n_lines=("Invoice", "size"), last_seen=("InvoiceDate", "max"))
      .reset_index()
      .sort_values(["Customer ID", "n_lines", "last_seen"], ascending=[True, False, False])
)
customers = (country_res.drop_duplicates("Customer ID")[["Customer ID", "Country"]]
             .rename(columns={"Customer ID": "customer_id", "Country": "country"})
             .reset_index(drop=True))

# Description: most frequent, ties broken alphabetically
desc_res = (
    df.groupby(["StockCode", "Description"]).size().reset_index(name="n_lines")
      .sort_values(["StockCode", "n_lines", "Description"], ascending=[True, False, True])
)
products = (desc_res.drop_duplicates("StockCode")[["StockCode", "Description"]]
            .rename(columns={"StockCode": "stock_code", "Description": "description"})
            .reset_index(drop=True))

display(Markdown(f"""
### Reference tables built
* `customers`: `{len(customers):,}` rows, unique key: `{customers['customer_id'].is_unique}`
* `products`: `{len(products):,}` rows, unique key: `{products['stock_code'].is_unique}`
"""))
display(products.loc[products["stock_code"] == "15058A"])


### Reference tables built
* `customers`: `5,880` rows, unique key: `True`
* `products`: `4,638` rows, unique key: `True`


,stock_code,description
25,15058A,BLUE WHITE SPOTS GARDEN PARASOL


**Results.** Both reference tables build cleanly: 5,880 customers and 4,638 products, each with a unique key confirmed by `is_unique`. `15058A` resolves to *BLUE WHITE SPOTS GARDEN PARASOL*, the 58-occurrence variant, as the tie-breaking rule intends.

## 1.2 Invoice-level Coherence

Before `invoice_no` can serve as a primary key, each invoice must map to exactly one customer and one timestamp.

In [5]:
check = df.groupby("Invoice").agg(
    n_customers=("Customer ID", "nunique"),
    n_dates=("InvoiceDate", "nunique"),
)

display(Markdown(f"""
### Invoice-level coherence
* Invoices linked to **multiple customers**: `{(check['n_customers'] > 1).sum():,}`
* Invoices carrying **multiple timestamps**: `{(check['n_dates'] > 1).sum():,}`
"""))


### Invoice-level coherence
* Invoices linked to **multiple customers**: `0`
* Invoices carrying **multiple timestamps**: `65`


**Results.** No invoice is linked to more than one customer, so the key is safe.

65 invoices carry more than one timestamp. Inspection shows these are lines of a single order recorded seconds apart rather than genuinely distinct events, so `MIN(invoice_date)` is used, corresponding to the time the order was placed.

In [6]:
invoices = (
    df.groupby("Invoice")
      .agg(customer_id=("Customer ID", "first"),
           invoice_date=("InvoiceDate", "min"),
           is_cancellation=("IsCancellation", "first"))
      .reset_index()
      .rename(columns={"Invoice": "invoice_no"})
)
invoices["is_cancellation"] = invoices["is_cancellation"].astype(int)

invoice_lines = (
    df[["Invoice", "StockCode", "Quantity", "Price", "LineRevenue"]]
      .rename(columns={"Invoice": "invoice_no", "StockCode": "stock_code",
                       "Quantity": "quantity", "Price": "price",
                       "LineRevenue": "line_revenue"})
      .reset_index(drop=True)
)
invoice_lines.insert(0, "line_id", invoice_lines.index + 1)

display(Markdown(f"""
### Transactional tables
* `invoices`: `{len(invoices):,}` rows, unique key: `{invoices['invoice_no'].is_unique}`
* `invoice_lines`: `{len(invoice_lines):,}` rows, unique key: `{invoice_lines['line_id'].is_unique}`
"""))


### Transactional tables
* `invoices`: `43,929` rows, unique key: `True`
* `invoice_lines`: `820,947` rows, unique key: `True`


**Results.** `invoices` holds 43,929 rows and `invoice_lines` 820,947, each with a unique key. The line count matches the input exactly, since normalisation splits the flat file by entity rather than aggregating it.

## 1.3 Integrity Control

Foreign keys are checked for orphans, and row count and revenue are checked for conservation, before anything is written to the database.

In [7]:
orphans = {
    "invoices -> customers": (~invoices["customer_id"].isin(customers["customer_id"])).sum(),
    "invoice_lines -> invoices": (~invoice_lines["invoice_no"].isin(invoices["invoice_no"])).sum(),
    "invoice_lines -> products": (~invoice_lines["stock_code"].isin(products["stock_code"])).sum(),
}

for link, n in orphans.items():
    status = "OK" if n == 0 else f"{n:,} orphans"
    display(Markdown(f"* `{link}`: **{status}**"))

display(Markdown(f"""
### Conservation check
* Rows: `{len(df):,}` in, `{len(invoice_lines):,}` out
* Net revenue: `£{df['LineRevenue'].sum():,.2f}` in, `£{invoice_lines['line_revenue'].sum():,.2f}` out
"""))

* `invoices -> customers`: **OK**

* `invoice_lines -> invoices`: **OK**

* `invoice_lines -> products`: **OK**


### Conservation check
* Rows: `820,947` in, `820,947` out
* Net revenue: `£16,728,372.22` in, `£16,728,372.22` out


**Results.** All three foreign-key relationships resolve with zero orphans. Both the row count (820,947) and total net revenue (£16,728,372.22) are conserved exactly between the flat file and the normalised tables.

Checking revenue alongside row count matters: a join error can preserve the number of rows while silently duplicating or dropping value.

# 2. Database Creation & Integrity Enforcement

Constraints are declared in the schema itself rather than trusted to upstream code: foreign keys, a `CHECK` constraint on the cancellation flag, and four indexes on the join and filter columns.

In [8]:
DDL = """
PRAGMA foreign_keys = ON;

DROP TABLE IF EXISTS invoice_lines;
DROP TABLE IF EXISTS invoices;
DROP TABLE IF EXISTS products;
DROP TABLE IF EXISTS customers;

CREATE TABLE customers (
    customer_id TEXT PRIMARY KEY,
    country     TEXT NOT NULL
);

CREATE TABLE products (
    stock_code  TEXT PRIMARY KEY,
    description TEXT
);

CREATE TABLE invoices (
    invoice_no      TEXT PRIMARY KEY,
    customer_id     TEXT NOT NULL,
    invoice_date    TIMESTAMP NOT NULL,
    is_cancellation INTEGER NOT NULL CHECK (is_cancellation IN (0, 1)),
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id)
);

CREATE TABLE invoice_lines (
    line_id      INTEGER PRIMARY KEY,
    invoice_no   TEXT NOT NULL,
    stock_code   TEXT NOT NULL,
    quantity     INTEGER NOT NULL,
    price        REAL NOT NULL,
    line_revenue REAL NOT NULL,
    FOREIGN KEY (invoice_no) REFERENCES invoices(invoice_no),
    FOREIGN KEY (stock_code) REFERENCES products(stock_code)
);

CREATE INDEX idx_invoices_customer ON invoices(customer_id);
CREATE INDEX idx_invoices_date     ON invoices(invoice_date);
CREATE INDEX idx_lines_invoice     ON invoice_lines(invoice_no);
CREATE INDEX idx_lines_product     ON invoice_lines(stock_code);
"""

if DB_PATH.exists():
    DB_PATH.unlink()

con = sqlite3.connect(DB_PATH)
con.executescript(DDL)
con.commit()

display(Markdown("Schema created with primary keys, foreign keys, a domain constraint and indexes."))

Schema created with primary keys, foreign keys, a domain constraint and indexes.

## 2.1 Loading

Tables are loaded in dependency order, parent before child, so that no foreign key points to a row that does not yet exist.

In [9]:

customers.to_sql("customers", con, if_exists="append", index=False)
products.to_sql("products", con, if_exists="append", index=False)
invoices.to_sql("invoices", con, if_exists="append", index=False)
invoice_lines.to_sql("invoice_lines", con, if_exists="append", index=False)
con.commit()

counts = pd.read_sql("""
    SELECT 'customers' AS table_name, COUNT(*) AS n FROM customers
    UNION ALL SELECT 'products',      COUNT(*) FROM products
    UNION ALL SELECT 'invoices',      COUNT(*) FROM invoices
    UNION ALL SELECT 'invoice_lines', COUNT(*) FROM invoice_lines
""", con)
display(counts)

,table_name,n
0,customers,5880
1,products,4638
2,invoices,43929
3,invoice_lines,820947


**Results.** The four tables load with the expected row counts: 5,880 customers, 4,638 products, 43,929 invoices and 820,947 invoice lines. Nothing was rejected during insertion.

## 2.2 Constraint Verification

Declaring a constraint is not the same as proving it fires. A deliberately invalid insert is attempted below.

In [10]:
con.execute("PRAGMA foreign_keys = ON")

violations = pd.read_sql("PRAGMA foreign_key_check", con)
display(Markdown(f"**Integrity violations detected by SQLite:** `{len(violations)}`"))

# Deliberately invalid insert: the database should reject it
try:
    con.execute("""INSERT INTO invoices (invoice_no, customer_id, invoice_date, is_cancellation)
                   VALUES ('TEST999', 'NONEXISTENT_CUSTOMER', '2011-01-01', 0)""")
    display(Markdown("The constraint did **not** fire."))
except sqlite3.IntegrityError as e:
    display(Markdown(f"Insert rejected by the database: `{e}` - the foreign key is enforced."))
finally:
    con.rollback()

**Integrity violations detected by SQLite:** `0`

Insert rejected by the database: `FOREIGN KEY constraint failed` - the foreign key is enforced.

**Results.** `PRAGMA foreign_key_check` returns 0 violations across the whole database.

The deliberately invalid insert, an invoice referencing a non-existent customer, is rejected with `FOREIGN KEY constraint failed`. The database refuses inconsistent data rather than merely happening to contain none.

# 3. Analytical Queries

Cancellations are retained in the database and netted in revenue aggregates: a cancelled line carries a negative quantity, so `SUM` yields net revenue directly. This matches the netting decision taken in notebook 03 for customer monetary value.

## 3.1 Monthly KPIs

In [11]:
q_monthly = """
SELECT
    strftime('%Y-%m', i.invoice_date)        AS month,
    COUNT(DISTINCT i.invoice_no)             AS n_orders,
    COUNT(DISTINCT i.customer_id)            AS n_active_customers,
    ROUND(SUM(l.line_revenue), 2)            AS net_revenue,
    ROUND(SUM(l.line_revenue)
          / COUNT(DISTINCT i.invoice_no), 2) AS avg_basket
FROM invoices i
JOIN invoice_lines l ON l.invoice_no = i.invoice_no
GROUP BY month
ORDER BY month
"""
monthly_kpi = pd.read_sql(q_monthly, con)
display(monthly_kpi.head(8))
print("Number of months:", len(monthly_kpi))

,month,n_orders,n_active_customers,net_revenue,avg_basket
0,2009-12,1871,1041,662078.61,353.86
1,2010-01,1190,744,528056.55,443.75
2,2010-02,1307,803,487928.19,373.32
3,2010-03,1855,1094,659363.19,355.45
4,2010-04,1577,989,578175.12,366.63
5,2010-05,1746,1059,558030.04,319.60
6,2010-06,1796,1083,606357.60,337.62
7,2010-07,1677,986,567103.16,338.17


Number of months: 25


**Results.** 25 monthly periods from December 2009 to December 2011.

December 2009 opens at £662,078.61 across 1,871 orders and 1,041 active customers. Average basket ranges from roughly £320 to £445 per order, consistent with a wholesale-oriented retailer rather than a consumer store.

The final month is incomplete, since the dataset stops on 9 December 2011. Any trend read from the last point is an artefact of the extract, not of the business.

## 3.2 Customer Ranking (Window Functions)

A single query combining a CTE, a join and four window functions: `RANK` globally, `RANK` partitioned by country, `NTILE` for deciles, and a running total for the concentration curve.

In [12]:
q_rank = """
WITH customer_revenue AS (
    SELECT
        i.customer_id,
        SUM(l.line_revenue)          AS total_revenue,
        COUNT(DISTINCT i.invoice_no) AS n_orders
    FROM invoices i
    JOIN invoice_lines l ON l.invoice_no = i.invoice_no
    GROUP BY i.customer_id
)
SELECT
    c.customer_id,
    cu.country,
    ROUND(c.total_revenue, 2) AS total_revenue,
    c.n_orders,
    RANK()    OVER (ORDER BY c.total_revenue DESC)                           AS global_rank,
    RANK()    OVER (PARTITION BY cu.country ORDER BY c.total_revenue DESC)   AS country_rank,
    NTILE(10) OVER (ORDER BY c.total_revenue DESC)                           AS decile,
    ROUND(100.0 * SUM(c.total_revenue) OVER (ORDER BY c.total_revenue DESC)
          / SUM(c.total_revenue) OVER (), 2)                                 AS cumulative_revenue_pct
FROM customer_revenue c
JOIN customers cu ON cu.customer_id = c.customer_id
ORDER BY global_rank
"""
customer_rank = pd.read_sql(q_rank, con)
display(customer_rank.head(10))

,customer_id,country,total_revenue,n_orders,global_rank,country_rank,decile,cumulative_revenue_pct
0,18102,United Kingdom,606243.25,147,1,1,1,3.62
1,14646,Netherlands,523202.74,153,2,1,1,6.75
2,14156,EIRE,299926.26,183,3,1,1,8.54
3,14911,EIRE,270203.05,469,4,2,1,10.16
4,17450,United Kingdom,235832.75,55,5,2,1,11.57
5,13694,United Kingdom,191196.18,158,6,3,1,12.71
6,17511,United Kingdom,171898.80,84,7,4,1,13.74
7,12415,Australia,143107.02,28,8,1,1,14.60
8,16684,United Kingdom,141530.29,64,9,5,1,15.44
9,15061,United Kingdom,136411.63,137,10,6,1,16.26


**Results.** The ranking surfaces two distinct high-value profiles at the top.

Customer `18102` (United Kingdom) reaches £606,243 across 147 orders, an average of roughly £4,100 per order. Customer `14911` (EIRE) reaches £270,203 across 469 orders, an average of roughly £576. Both sit in the top revenue tier for opposite reasons, one through basket size and the other through frequency.

This is precisely the distinction that RFM segmentation formalises in notebook 03: revenue alone cannot separate these two behaviours.

In [13]:
top_decile = customer_rank.loc[customer_rank["decile"] == 1, "total_revenue"].sum()
total_revenue = customer_rank["total_revenue"].sum()
n_customers_half = (customer_rank["cumulative_revenue_pct"] <= 50).sum()

display(Markdown(f"""
### Revenue concentration
* The **top decile** ({(customer_rank['decile'] == 1).sum():,} customers) generates **{100*top_decile/total_revenue:.1f}%** of net revenue.
* **{n_customers_half:,} customers** ({100*n_customers_half/len(customer_rank):.1f}% of the base) account for **half** of total revenue.
"""))


### Revenue concentration
* The **top decile** (588 customers) generates **63.4%** of net revenue.
* **271 customers** (4.6% of the base) account for **half** of total revenue.


**Results.** The top decile, 588 customers, generates 63.4% of net revenue. 271 customers, 4.6% of the base, account for half of it.

This is steeper than a standard Pareto split, and it is the single most actionable figure in this notebook: retention effort concentrated on a few hundred accounts protects a disproportionate share of revenue.

## 3.3 Purchase Intervals (`LAG`)

Each customer's ordered purchase sequence is reconstructed to measure the gap between consecutive orders. Cancellations are excluded here, since a cancelled invoice is not a purchase event.

In [14]:
q_lag = """
WITH orders AS (
    SELECT DISTINCT
        i.customer_id,
        i.invoice_no,
        DATE(i.invoice_date) AS order_date
    FROM invoices i
    WHERE i.is_cancellation = 0
),
sequenced AS (
    SELECT
        customer_id,
        invoice_no,
        order_date,
        ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY order_date) AS order_rank,
        LAG(order_date) OVER (PARTITION BY customer_id ORDER BY order_date) AS previous_date
    FROM orders
)
SELECT
    customer_id,
    invoice_no,
    order_date,
    order_rank,
    JULIANDAY(order_date) - JULIANDAY(previous_date) AS gap_days
FROM sequenced
ORDER BY customer_id, order_rank
"""
seq = pd.read_sql(q_lag, con)
display(seq.head(12))

,customer_id,invoice_no,order_date,order_rank,gap_days
0,12346,499763,2010-03-02,1,NaN
1,12346,513774,2010-06-28,2,118.0
2,12346,541431,2011-01-18,3,204.0
3,12347,529924,2010-10-31,1,NaN
4,12347,537626,2010-12-07,2,37.0
5,12347,542237,2011-01-26,3,50.0
6,12347,549222,2011-04-07,4,71.0
7,12347,556201,2011-06-09,5,63.0
8,12347,562032,2011-08-02,6,54.0
9,12347,573511,2011-10-31,7,90.0


**Results.** The reconstruction works as intended. Customer `12346` shows three orders spaced 118 and 204 days apart, an occasional buyer. Customer `12347` shows eight orders with gaps between 37 and 90 days, a regular one. `NaN` on the first order of each customer is expected, since `LAG` has no prior row to reference.

In [15]:
gaps = seq["gap_days"].dropna()
single_order = seq.groupby("customer_id")["order_rank"].max().eq(1).sum()

display(Markdown(f"""
### Repurchase rhythm
* Customers with a **single order**: `{single_order:,}` out of `{seq['customer_id'].nunique():,}` ({100*single_order/seq['customer_id'].nunique():.1f}%)
* Median gap between consecutive orders: **`{gaps.median():.0f}` days**
* Mean: `{gaps.mean():.0f}` days - the gap to the median measures skewness
* 90th percentile: `{gaps.quantile(0.90):.0f}` days
"""))


### Repurchase rhythm
* Customers with a **single order**: `1,620` out of `5,854` (27.7%)
* Median gap between consecutive orders: **`25` days**
* Mean: `52` days - the gap to the median measures skewness
* 90th percentile: `136` days


**Results.** 1,620 of 5,854 customers (27.7%) never placed a second order. More than one buyer in four was acquired and never returned.

Among those who did return, the median gap between consecutive orders is 25 days, with a heavily right-skewed distribution: mean 52 days, 90th percentile 136. The gap between mean and median confirms a mix of frequent buyers and occasional ones rather than a homogeneous population.

**Cross-validation.** These figures reproduce the pandas-based calculation in notebook 04 (median 25, mean 51.8, p90 136) to within one customer out of 5,854. The difference is attributable to date truncation in SQL against full timestamps in pandas. Two independent implementations converge.

## 3.4 BI Extracts

Four aggregates are exported for the Power BI dashboard: monthly KPIs, customer ranking, product performance and country breakdown.

In [16]:
EXPORT = PROC / "powerbi"
EXPORT.mkdir(exist_ok=True)

monthly_kpi.to_csv(EXPORT / "monthly_kpi.csv", index=False)
customer_rank.to_csv(EXPORT / "customer_rank.csv", index=False)

q_products = """
SELECT p.stock_code, p.description,
       SUM(l.quantity)               AS units,
       ROUND(SUM(l.line_revenue), 2) AS net_revenue,
       COUNT(DISTINCT i.customer_id) AS n_customers
FROM invoice_lines l
JOIN invoices i ON i.invoice_no = l.invoice_no
JOIN products p ON p.stock_code = l.stock_code
GROUP BY p.stock_code, p.description
ORDER BY net_revenue DESC
"""
pd.read_sql(q_products, con).to_csv(EXPORT / "products.csv", index=False)

q_countries = """
SELECT cu.country,
       COUNT(DISTINCT cu.customer_id) AS n_customers,
       ROUND(SUM(l.line_revenue), 2)  AS net_revenue
FROM customers cu
JOIN invoices i      ON i.customer_id = cu.customer_id
JOIN invoice_lines l ON l.invoice_no = i.invoice_no
GROUP BY cu.country
ORDER BY net_revenue DESC
"""
pd.read_sql(q_countries, con).to_csv(EXPORT / "countries.csv", index=False)

print("Files exported:", [f.name for f in EXPORT.glob("*.csv")])
con.close()

Files exported: ['countries.csv', 'customer_rank.csv', 'monthly_kpi.csv', 'products.csv']


**Results.** Four CSV files are written to `data/processed/powerbi/`, feeding the four data sources of the dashboard.

# 4. Summary

**What this notebook establishes.** The 820,947-row flat file is decomposed into four normalised tables with primary keys, foreign keys, a domain constraint and indexes declared in SQLite. Integrity is verified before loading and confirmed afterwards by a rejected insert. Two remediation rules were required, for multi-country customers and multi-label products, each documented with its rationale.

**Key findings.**

| Metric | Value |
|---|---|
| Revenue from the top customer decile | 63.4% |
| Customers generating half of revenue | 271 (4.6%) |
| Customers with a single order | 27.7% |
| Median interval between orders | 25 days |
| Mean interval between orders | 52 days |

**Limitations.**

Country is resolved to a single value per customer, which flattens genuinely multi-country buyers (12 cases). Product descriptions are collapsed to one label per stock code, discarding variant wording that could carry catalogue history. Neither affects the aggregates above, since both fields are descriptive rather than measured.

The December 2011 figures cover nine days only and are not comparable to full months.

**Downstream use.** `retail.db` supports ad-hoc querying, and the CSV extracts feed the Power BI dashboard.